In [3]:

# ============================================================
# PROJECT GENOME: Autonomous Scientific Discovery Engine
# One-cell Google Colab application
# ============================================================

import os
import sys
import time
import socket
import subprocess
import textwrap

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "streamlit", "pandas", "numpy", "networkx",
    "plotly", "scikit-learn", "requests"
])

APP = r'''
import json
import re
import numpy as np
import pandas as pd
import networkx as nx
import requests
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

st.set_page_config(page_title="PROJECT GENOME", layout="wide")

st.markdown("""
<style>
.stApp {
 background: linear-gradient(135deg,#07121e,#102439,#112f36);
 color: #e7f1ff;
}
[data-testid="stMetric"] {
 background: rgba(24,48,69,.88);
 border: 1px solid #31546b;
 padding: 14px;
 border-radius: 12px;
}
h1,h2,h3 {color:#d9efff;}
[data-testid="stSidebar"] {background:#091725;}
</style>
""", unsafe_allow_html=True)

# ------------------------------------------------------------
# 1. DEMONSTRATION DATA
# All built-in relationships are synthetic examples.
# ------------------------------------------------------------

nodes = [
    ("TP53","Gene"),("BRCA1","Gene"),("EGFR","Gene"),
    ("MYC","Gene"),("IL6","Gene"),("TNF","Gene"),
    ("KRAS","Gene"),("PTEN","Gene"),("PIK3CA","Gene"),
    ("STAT3","Gene"),("Apoptosis","Pathway"),
    ("DNA Repair","Pathway"),("PI3K-AKT","Pathway"),
    ("MAPK","Pathway"),("Inflammation","Pathway"),
    ("Cancer","Disease"),("Inflammatory Disease","Disease"),
    ("Study A","Paper"),("Study B","Paper"),
    ("Study C","Paper"),("Study D","Paper")
]

node_df = pd.DataFrame(nodes, columns=["id","type"])

relationships = [
    ("TP53","Apoptosis","associated_with",.91,"Study A"),
    ("TP53","DNA Repair","associated_with",.84,"Study A"),
    ("BRCA1","DNA Repair","associated_with",.95,"Study B"),
    ("EGFR","MAPK","associated_with",.87,"Study B"),
    ("KRAS","MAPK","associated_with",.82,"Study C"),
    ("PTEN","PI3K-AKT","associated_with",.89,"Study C"),
    ("PIK3CA","PI3K-AKT","associated_with",.85,"Study D"),
    ("IL6","Inflammation","associated_with",.88,"Study A"),
    ("TNF","Inflammation","associated_with",.90,"Study B"),
    ("STAT3","Inflammation","associated_with",.72,"Study D"),
    ("MYC","Apoptosis","associated_with",.65,"Study C"),
    ("Apoptosis","Cancer","related_to",.76,"Study A"),
    ("DNA Repair","Cancer","related_to",.83,"Study B"),
    ("MAPK","Cancer","related_to",.86,"Study C"),
    ("PI3K-AKT","Cancer","related_to",.84,"Study D"),
    ("Inflammation","Inflammatory Disease","related_to",.79,"Study B"),
    ("TP53","Cancer","associated_with",.74,"Study A"),
    ("EGFR","Cancer","associated_with",.81,"Study B"),
    ("IL6","Inflammatory Disease","associated_with",.77,"Study D"),
    ("BRCA1","TP53","shared_context",.62,"Study A"),
    ("KRAS","PIK3CA","shared_context",.61,"Study C")
]

edge_df = pd.DataFrame(
    relationships,
    columns=["source","target","relation","weight","evidence_id"]
)

G = nx.Graph()

for row in node_df.itertuples(index=False):
    G.add_node(row.id, node_type=row.type)

for row in edge_df.itertuples(index=False):
    G.add_edge(
        row.source, row.target,
        relation=row.relation,
        weight=float(row.weight),
        evidence_id=row.evidence_id
    )

# ------------------------------------------------------------
# 2. KNOWLEDGE GRAPH VISUALIZATION
# ------------------------------------------------------------

def graph_figure(graph, focus=None):
    pos = nx.spring_layout(graph, seed=23, k=.9)

    ex, ey = [], []
    for u, v in graph.edges():
        ex += [pos[u][0], pos[v][0], None]
        ey += [pos[u][1], pos[v][1], None]

    edge_trace = go.Scatter(
        x=ex, y=ey, mode="lines",
        line=dict(width=1.2, color="#55728c"),
        hoverinfo="none"
    )

    palette = {
        "Gene":"#42d6a4",
        "Pathway":"#5da9ff",
        "Disease":"#f59e72",
        "Paper":"#c4a7ff"
    }

    names = list(graph.nodes())
    colors = [
        "#ff5252" if n == focus
        else palette.get(graph.nodes[n].get("node_type"), "#aaaaaa")
        for n in names
    ]

    node_trace = go.Scatter(
        x=[pos[n][0] for n in names],
        y=[pos[n][1] for n in names],
        mode="markers+text",
        text=names,
        textposition="bottom center",
        marker=dict(size=17, color=colors, line=dict(width=1,color="white")),
        hovertext=[
            f"{n}<br>Type: {graph.nodes[n].get('node_type')}<br>"
            f"Connections: {graph.degree(n)}"
            for n in names
        ],
        hoverinfo="text"
    )

    fig = go.Figure([edge_trace,node_trace])
    fig.update_layout(
        height=620,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font_color="#e7f1ff",
        showlegend=False,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        margin=dict(l=10,r=10,t=30,b=10)
    )
    return fig

# ------------------------------------------------------------
# 3. GRAPH-DERIVED HYPOTHESIS GENERATOR
# These are candidates, not validated discoveries.
# ------------------------------------------------------------

def generate_hypotheses(graph, node_table):
    genes = node_table.loc[node_table.type=="Gene","id"].tolist()
    pathways = node_table.loc[
        node_table.type=="Pathway","id"
    ].tolist()

    results = []

    for gene in genes:
        for pathway in pathways:
            if graph.has_edge(gene,pathway):
                continue

            shared = set(graph.neighbors(gene)) & set(
                graph.neighbors(pathway)
            )

            if not shared:
                continue

            weights = []
            for middle in shared:
                weights += [
                    graph[gene][middle]["weight"],
                    graph[pathway][middle]["weight"]
                ]

            mean_weight = float(np.mean(weights))
            common = len(shared)

            score = (
                .40 * min(common/3,1)
                + .35 * mean_weight
                + .15 * min(graph.degree(gene)/5,1)
                + .10 * min(graph.degree(pathway)/5,1)
            ) * 100

            results.append({
                "Gene":gene,
                "Candidate pathway":pathway,
                "Shared neighbors":", ".join(sorted(shared)),
                "Common-neighbor count":common,
                "Mean neighboring edge weight":round(mean_weight,3),
                "Prioritization score":round(score,2),
                "Status":"Graph-derived hypothesis; validation required"
            })

    return pd.DataFrame(results).sort_values(
        "Prioritization score",ascending=False
    ).reset_index(drop=True)

hypotheses = generate_hypotheses(G,node_df)

# ------------------------------------------------------------
# 4. CONNECTION EXPLORER
# ------------------------------------------------------------

def find_connection(source,target):
    try:
        path = nx.shortest_path(G,source,target)
    except (nx.NetworkXNoPath,nx.NodeNotFound):
        return []

    result = []
    for a,b in zip(path[:-1],path[1:]):
        info = G[a][b]
        result.append({
            "source":a,
            "target":b,
            "relation":info["relation"],
            "weight":info["weight"],
            "evidence_id":info["evidence_id"]
        })
    return result

# ------------------------------------------------------------
# 5. OPTIONAL LIVE PUBMED SEARCH
# ------------------------------------------------------------

def pubmed_search(query, limit=10):
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    headers = {"User-Agent":"PROJECT-GENOME-research-prototype/1.0"}

    r = requests.get(
        base+"esearch.fcgi",
        params={
            "db":"pubmed","term":query,
            "retmax":limit,"retmode":"json","sort":"relevance"
        },
        headers=headers,timeout=20
    )
    r.raise_for_status()
    ids = r.json()["esearchresult"]["idlist"]

    if not ids:
        return pd.DataFrame(columns=["pmid","title","date","url"])

    r = requests.get(
        base+"esummary.fcgi",
        params={
            "db":"pubmed","id":",".join(ids),"retmode":"json"
        },
        headers=headers,timeout=20
    )
    r.raise_for_status()
    payload = r.json()["result"]

    rows = []
    for pmid in ids:
        item = payload.get(pmid,{})
        rows.append({
            "pmid":pmid,
            "title":re.sub(r"<[^>]+>","",item.get("title","")),
            "date":item.get("pubdate",""),
            "url":"https://pubmed.ncbi.nlm.nih.gov/"+pmid+"/"
        })

    return pd.DataFrame(rows)

# ------------------------------------------------------------
# 6. TEXT SIMILARITY DEMONSTRATION
# ------------------------------------------------------------

demo_docs = pd.DataFrame([
    {
        "title":"Synthetic example: DNA repair",
        "text":"DNA repair genome integrity cellular damage tumor suppressor"
    },
    {
        "title":"Synthetic example: inflammation",
        "text":"Inflammation cytokine immune response inflammatory disease"
    },
    {
        "title":"Synthetic example: growth signaling",
        "text":"Cell growth MAPK PI3K AKT signaling pathways cancer"
    },
    {
        "title":"Synthetic example: apoptosis",
        "text":"Apoptosis programmed cell death TP53 cell fate"
    },
    {
        "title":"Synthetic example: transcription",
        "text":"Transcription gene expression STAT3 MYC regulation"
    }
])

def rank_text(query, documents):
    corpus = [query] + documents["text"].fillna("").tolist()
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(corpus)
    similarities = cosine_similarity(matrix[0:1],matrix[1:]).ravel()
    result = documents.copy()
    result["similarity"] = similarities
    return result.sort_values("similarity",ascending=False)

# ------------------------------------------------------------
# 7. EXPLORATORY MACHINE LEARNING
# Synthetic labels derived from the heuristic score.
# ------------------------------------------------------------

@st.cache_resource
def train_demo_model():
    if len(hypotheses) < 8:
        return None, {}

    X = hypotheses[
        ["Common-neighbor count","Mean neighboring edge weight"]
    ].astype(float)

    y = (
        hypotheses["Prioritization score"]
        >= hypotheses["Prioritization score"].median()
    ).astype(int)

    if y.nunique() < 2:
        return None, {}

    X_train,X_test,y_train,y_test = train_test_split(
        X,y,test_size=.3,random_state=42,stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=120,
        max_depth=4,
        class_weight="balanced",
        random_state=42
    )
    model.fit(X_train,y_train)
    prediction = model.predict(X_test)
    probability = model.predict_proba(X_test)[:,1]

    metrics = {
        "Accuracy":accuracy_score(y_test,prediction),
        "F1":f1_score(y_test,prediction,zero_division=0)
    }

    if y_test.nunique()==2:
        metrics["ROC-AUC"] = roc_auc_score(y_test,probability)

    return model,metrics

model,model_metrics = train_demo_model()

# ------------------------------------------------------------
# 8. DASHBOARD NAVIGATION
# ------------------------------------------------------------

st.sidebar.title("PROJECT GENOME")
st.sidebar.caption("Autonomous Scientific Discovery Engine")

page = st.sidebar.radio(
    "Navigate",
    [
        "Overview","Knowledge Graph","Hypothesis Engine",
        "Connection Explorer","PubMed Search",
        "Text Similarity","ML Lab","Export Center"
    ]
)

st.title("PROJECT GENOME")
st.subheader("Explore biological connections and research hypotheses")

st.warning(
    "The built-in graph, edge weights, and example text records are "
    "synthetic. They are not validated biological evidence. PubMed search "
    "retrieves publication records but does not prove a scientific claim."
)

# ------------------------------------------------------------
# 9. OVERVIEW
# ------------------------------------------------------------

if page=="Overview":
    c1,c2,c3,c4 = st.columns(4)
    c1.metric("Graph nodes",len(G.nodes()))
    c2.metric("Relationships",len(G.edges()))
    c3.metric("Candidate hypotheses",len(hypotheses))
    c4.metric("Node categories",node_df.type.nunique())

    st.plotly_chart(graph_figure(G),use_container_width=True)

    st.subheader("How discovery works")
    st.code(
        "Research question\n"
        "  -> Knowledge graph\n"
        "  -> Candidate generation\n"
        "  -> Literature retrieval\n"
        "  -> Evidence assessment\n"
        "  -> Independent validation",
        language="text"
    )

    st.dataframe(
        node_df.groupby("type").size().reset_index(name="count"),
        use_container_width=True
    )

# ------------------------------------------------------------
# 10. KNOWLEDGE GRAPH
# ------------------------------------------------------------

elif page=="Knowledge Graph":
    types = st.multiselect(
        "Node categories",
        sorted(node_df.type.unique()),
        default=sorted(node_df.type.unique())
    )

    allowed = set(node_df.loc[node_df.type.isin(types),"id"])
    subgraph = G.subgraph(allowed).copy()

    st.plotly_chart(graph_figure(subgraph),use_container_width=True)
    st.subheader("Nodes")
    st.dataframe(
        node_df[node_df.id.isin(allowed)],use_container_width=True
    )
    st.subheader("Relationships")
    st.dataframe(
        edge_df[
            edge_df.source.isin(allowed)
            & edge_df.target.isin(allowed)
        ],
        use_container_width=True
    )

# ------------------------------------------------------------
# 11. HYPOTHESIS ENGINE
# ------------------------------------------------------------

elif page=="Hypothesis Engine":
    st.write(
        "Candidate scores prioritize missing gene-pathway links based on "
        "shared neighbors and graph properties. They are not probabilities."
    )

    threshold = st.slider("Minimum score",0,100,0)
    filtered = hypotheses[
        hypotheses["Prioritization score"]>=threshold
    ]

    st.dataframe(filtered,use_container_width=True)

    if len(filtered):
        idx = st.selectbox(
            "Inspect candidate",
            filtered.index.tolist(),
            format_func=lambda i:
                filtered.loc[i,"Gene"]+" -> "+
                filtered.loc[i,"Candidate pathway"]
        )
        row = filtered.loc[idx]

        st.subheader(row["Gene"]+" -> "+row["Candidate pathway"])
        st.write("Shared neighbors:",row["Shared neighbors"])
        st.write("Prioritization score:",row["Prioritization score"])
        st.write("Status:",row["Status"])

        st.info(
            "Validate this candidate using real publications, appropriate "
            "controls, and independent biological datasets."
        )

    st.plotly_chart(
        px.histogram(
            hypotheses,x="Prioritization score",nbins=12,
            title="Candidate score distribution"
        ),
        use_container_width=True
    )

# ------------------------------------------------------------
# 12. CONNECTION EXPLORER
# ------------------------------------------------------------

elif page=="Connection Explorer":
    all_nodes = node_df.id.tolist()

    source = st.selectbox(
        "Starting concept",all_nodes,
        index=all_nodes.index("TP53")
    )
    target = st.selectbox(
        "Target concept",all_nodes,
        index=all_nodes.index("Cancer")
    )

    if st.button("Find connection"):
        st.session_state["path_result"] = find_connection(source,target)
        st.session_state["path_nodes"] = (source,target)

    if "path_result" in st.session_state:
        result = st.session_state["path_result"]
        start,end = st.session_state["path_nodes"]

        if start==end:
            st.write("The selected concepts are identical.")
        elif not result:
            st.warning("No connection exists in the current graph.")
        else:
            path_nodes = [result[0]["source"]]
            path_nodes += [r["target"] for r in result]
            st.subheader("Shortest path")
            st.write(" -> ".join(path_nodes))
            st.dataframe(pd.DataFrame(result),use_container_width=True)
            st.caption(
                "Connectivity in this demonstration graph is not proof "
                "of causality or a validated biological mechanism."
            )

            st.plotly_chart(
                graph_figure(G,focus=start),use_container_width=True
            )

# ------------------------------------------------------------
# 13. PUBMED SEARCH
# ------------------------------------------------------------

elif page=="PubMed Search":
    st.subheader("Search real publication records")
    query = st.text_input("Gene, pathway, disease, or research question",
                          "TP53 DNA repair")
    limit = st.slider("Maximum results",5,30,10)

    if st.button("Search PubMed"):
        try:
            with st.spinner("Querying NCBI PubMed..."):
                results = pubmed_search(query,limit)
            st.session_state["pubmed_results"] = results
            st.session_state["pubmed_query"] = query
        except Exception as e:
            st.error("Search failed. Check internet access. Details: "+str(e))

    if "pubmed_results" in st.session_state:
        results = st.session_state["pubmed_results"]
        st.write("Query:",st.session_state["pubmed_query"])
        st.dataframe(results,use_container_width=True)

        st.download_button(
            "Download publication records",
            data=results.to_csv(index=False).encode("utf-8"),
            file_name="genome_pubmed_results.csv",
            mime="text/csv"
        )

        st.caption(
            "Review publication abstracts and study methods before using "
            "a paper as evidence for a biological relationship."
        )

# ------------------------------------------------------------
# 14. TEXT SIMILARITY
# ------------------------------------------------------------

elif page=="Text Similarity":
    query = st.text_input(
        "Research question",
        "How does DNA repair relate to cell death?"
    )
    ranked = rank_text(query,demo_docs)
    st.dataframe(ranked,use_container_width=True)

    st.caption(
        "Built-in texts are synthetic. TF-IDF measures lexical similarity, "
        "not scientific validity."
    )

# ------------------------------------------------------------
# 15. ML LAB
# ------------------------------------------------------------

elif page=="ML Lab":
    st.subheader("Exploratory Random Forest model")

    if model is None:
        st.info("Not enough candidate hypotheses to train the demo model.")
    else:
        cols = st.columns(len(model_metrics))
        for col,(name,value) in zip(cols,model_metrics.items()):
            col.metric(name,f"{value:.3f}")

        st.caption(
            "These metrics evaluate a synthetic label created from the "
            "heuristic score. They do not establish biological accuracy."
        )

        importance = pd.DataFrame({
            "Feature":[
                "Common-neighbor count",
                "Mean neighboring edge weight"
            ],
            "Importance":model.feature_importances_
        })

        st.plotly_chart(
            px.bar(
                importance,x="Importance",y="Feature",
                orientation="h",title="Feature importance"
            ),
            use_container_width=True
        )

        a,b = st.columns(2)
        with a:
            common = st.number_input(
                "Common-neighbor count",0,20,2
            )
        with b:
            weight = st.slider(
                "Mean neighboring edge weight",0.0,1.0,.75,.01
            )

        sample = pd.DataFrame([{
            "Common-neighbor count":common,
            "Mean neighboring edge weight":weight
        }])

        score = model.predict_proba(sample)[0,1]
        st.metric("Synthetic priority-class score",f"{score:.3f}")

# ------------------------------------------------------------
# 16. EXPORT CENTER
# ------------------------------------------------------------

elif page=="Export Center":
    st.subheader("Download research artifacts")

    st.download_button(
        "Download nodes CSV",
        data=node_df.to_csv(index=False).encode("utf-8"),
        file_name="genome_nodes.csv",
        mime="text/csv"
    )

    st.download_button(
        "Download relationships CSV",
        data=edge_df.to_csv(index=False).encode("utf-8"),
        file_name="genome_relationships.csv",
        mime="text/csv"
    )

    st.download_button(
        "Download hypotheses CSV",
        data=hypotheses.to_csv(index=False).encode("utf-8"),
        file_name="genome_hypotheses.csv",
        mime="text/csv"
    )

    bundle = {
        "project":"PROJECT GENOME",
        "data_status":"Synthetic demonstration graph",
        "nodes":node_df.to_dict(orient="records"),
        "relationships":edge_df.to_dict(orient="records"),
        "hypotheses":hypotheses.to_dict(orient="records"),
        "limitations":[
            "Built-in relationships are synthetic.",
            "Graph connectivity does not establish causality.",
            "Prioritization scores are not probabilities.",
            "Literature findings require independent assessment."
        ]
    }

    st.download_button(
        "Download complete JSON bundle",
        data=json.dumps(bundle,indent=2).encode("utf-8"),
        file_name="genome_bundle.json",
        mime="application/json"
    )

st.divider()
st.caption(
    "PROJECT GENOME | Bioinformatics research prototype | "
    "Synthetic graph associations require validation"
)
'''

# ============================================================
# SAVE APPLICATION AND START SERVER ONCE
# ============================================================

with open("app.py", "w", encoding="utf-8") as f:
    f.write(textwrap.dedent(APP))

print("Application saved to app.py")

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.bind(("0.0.0.0", 0))
    PORT = sock.getsockname()[1]

log = open("genome_streamlit.log", "w", encoding="utf-8")

process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", str(PORT),
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false"
    ],
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(8)

if process.poll() is not None:
    log.close()
    with open("genome_streamlit.log", encoding="utf-8") as f:
        print(f.read()[-12000:])
    raise RuntimeError("Streamlit failed to start; see log above.")

print("Streamlit is running on port", PORT)

# Public tunnel for Google Colab, if ngrok is configured.
try:
    from pyngrok import ngrok

    token = os.environ.get("NGROK_AUTH_TOKEN")
    if token:
        ngrok.set_auth_token(token)

    url = ngrok.connect(PORT, "http")
    print("\nPROJECT GENOME dashboard:")
    print(url)
    print("Keep this cell running while using the dashboard.")

except Exception as exc:
    print("Local URL: http://localhost:" + str(PORT))
    print("A public tunnel could not be created.")
    print("In Colab, configure a valid NGROK_AUTH_TOKEN if needed.")
    print("Tunnel details:", exc)


Application saved to app.py
Streamlit is running on port 44765
Local URL: http://localhost:44765
A public tunnel could not be created.
In Colab, configure a valid NGROK_AUTH_TOKEN if needed.
Tunnel details: No module named 'pyngrok'
